# Python으로 Redis 사용하기 — 자료구조

`redis-py`를 사용해 String, List, Set, Sorted Set, Hash를 Python 코드로 다룹니다.  
각 셀을 **순서대로** 실행하세요.

## 0. 사전 준비

### 개발환경 구축 
```shell
uv sync
```

### 환경변수 설정 

`.env.sample`을 복사해 `.env` 파일을 만들고 Redis 연결 정보를 입력하세요.

```
REDIS_HOST=localhost
REDIS_PORT=6379
REDIS_DB=0
REDIS_PASSWORD=
```

In [1]:
import os
import redis
import pandas as pd
from dotenv import load_dotenv

# .env 파일에서 환경변수(REDIS_HOST, PORT 등)를 불러옵니다
load_dotenv()

# Redis 클라이언트 생성
# decode_responses=True: Redis가 반환하는 bytes 값을 자동으로 str로 변환해줍니다
#   → 없으면 r.get('key') 결과가 b'value' 형태의 바이트로 옵니다
r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),   # Redis 서버 주소 (기본값: localhost)
    port=int(os.getenv('REDIS_PORT', 6379)),     # Redis 포트 (기본값: 6379)
    db=int(os.getenv('REDIS_DB', 0)),            # 사용할 DB 번호 (0~15, 기본값: 0)
    password=os.getenv('REDIS_PASSWORD') or None, # 비밀번호 없으면 None 전달
    decode_responses=True,
)

# 서버 연결 확인 — True가 출력되면 정상 연결된 것입니다
print(r.ping())  # True

True


## 1. String

| Redis 명령어 | Python 메서드 |
|---|---|
| `SET key value` | `r.set(key, value)` |
| `GET key` | `r.get(key)` |
| `SET key value EX n` | `r.set(key, value, ex=n)` |
| `TTL key` | `r.ttl(key)` |
| `INCR key` | `r.incr(key)` |
| `INCRBY key n` | `r.incrby(key, n)` |

> `decode_responses=True` 덕분에 값이 `bytes` 대신 `str`로 반환됩니다.

In [2]:
# r.set(key, value): Redis SET 명령어와 동일
r.set('product:name', 'Redis 입문서')

# r.get(key):        Redis GET 명령어와 동일
print(r.get('product:name'))       # Redis 입문서

Redis 입문서


In [3]:
# 만료 시간(TTL) 설정 — ex=초 단위로 자동 삭제 시간을 지정합니다
# 세션 토큰·인증 코드처럼 일정 시간 후 만료돼야 하는 데이터에 활용합니다
r.set('session:user7', 'active', ex=3600)   # 1시간(3600초) 후 자동 삭제

# r.ttl(key): 키가 삭제되기까지 남은 초를 반환 (-1이면 만료 없음, -2면 키 없음)
print(r.ttl('session:user7'))      # ~3600

3600


In [4]:
# 카운터 — r.incr / r.incrby 로 원자적(atomic)으로 숫자를 증가시킵니다
# 원자적이란: 여러 요청이 동시에 들어와도 값이 꼬이지 않고 정확하게 증가함을 의미합니다
# 페이지 조회수, 좋아요 수처럼 빠른 카운팅이 필요한 경우에 적합합니다
r.set('views:100', 0)          # 초기값 0으로 설정
r.incr('views:100')             # +1 증가 → 1
r.incrby('views:100', 4)        # +4 증가 → 5

print(r.get('views:100'))       # 5

5


## 2. List

| Redis 명령어 | Python 메서드 |
|---|---|
| `RPUSH key v1 v2` | `r.rpush(key, v1, v2)` |
| `LRANGE key 0 -1` | `r.lrange(key, 0, -1)` |
| `LTRIM key s e` | `r.ltrim(key, s, e)` |
| `LPOP key` | `r.lpop(key)` |
| `LLEN key` | `r.llen(key)` |

In [5]:
r.delete('recent:user7')                                     # 이전 데이터 초기화

# r.rpush: 리스트 오른쪽(꼬리)에 항목 추가 — 시청/방문 순서가 보존됩니다
r.rpush('recent:user7', 'item:30', 'item:12', 'item:55', 'item:88')  # 4개 추가

# r.ltrim(key, start, end): start~end 인덱스만 남기고 나머지 삭제
#   → -3은 끝에서 3번째, -1은 마지막 요소를 의미합니다 (Python 슬라이싱과 동일)
r.ltrim('recent:user7', -3, -1)                              # 마지막 3개만 유지

print(r.lrange('recent:user7', 0, -1))   # ['item:12', 'item:55', 'item:88']

['item:12', 'item:55', 'item:88']


In [6]:
r.delete('queue:jobs')

# rpush로 오른쪽에 추가하고, lpop으로 왼쪽에서 꺼내면 FIFO 순서가 됩니다
# 이메일 발송, 이미지 처리 같은 비동기 작업 처리에 자주 쓰입니다
r.rpush('queue:jobs', 'job:1', 'job:2', 'job:3')  # 작업 3개를 큐에 추가
job = r.lpop('queue:jobs')                         # 가장 먼저 들어온 작업 꺼내기

print(f'처리 중: {job}')                           # 처리 중: job:1

처리 중: job:1


In [7]:
# r.lrange(key, 0, -1): 리스트 전체 조회 (0=처음, -1=끝)
remaining = r.lrange('queue:jobs', 0, -1)
print(f'남은 작업: {remaining}')          # ['job:2', 'job:3']

남은 작업: ['job:2', 'job:3']


## 3. Set

| Redis 명령어 | Python 메서드 |
|---|---|
| `SADD key v1 v2` | `r.sadd(key, v1, v2)` |
| `SMEMBERS key` | `r.smembers(key)` |
| `SISMEMBER key v` | `r.sismember(key, v)` |
| `SCARD key` | `r.scard(key)` |
| `SINTER k1 k2` | `r.sinter(k1, k2)` |
| `SDIFF k1 k2` | `r.sdiff(k1, k2)` |

In [8]:
r.delete('likes:post:100')

# r.sadd(key, *values): 하나 이상의 값을 집합에 추가
r.sadd('likes:post:100', 'user:1', 'user:3', 'user:7')
print(r.smembers('likes:post:100'))               # {'user:1', 'user:3', 'user:7'}

# r.scard(key):         집합의 원소 개수 반환 (좋아요 수)
print(r.scard('likes:post:100'))                  # 3  (좋아요 수)

# r.sismember(key, v):  특정 값이 집합에 있는지 확인 (있으면 1/True)
print(r.sismember('likes:post:100', 'user:3'))    # True (user:3이 좋아요를 눌렀는지 확인)

{'user:1', 'user:7', 'user:3'}
3
1


In [9]:
r.delete('interests:user1', 'interests:user2')

r.sadd('interests:user1', 'python', 'redis', 'sql')
r.sadd('interests:user2', 'redis', 'sql', 'docker')

# r.sinter(k1, k2): 교집합 — 두 집합 모두에 있는 원소
print('교집합:', r.sinter('interests:user1', 'interests:user2'))   # {'redis', 'sql'}

# r.sdiff(k1, k2):  차집합 — k1에만 있고 k2에는 없는 원소
# 활용 예: "당신과 비슷한 관심사를 가진 사용자", "팔로우 추천" 기능 등
print('차집합:', r.sdiff('interests:user1', 'interests:user2'))    # {'python'}

교집합: {'redis', 'sql'}
차집합: {'python'}


## 4. Sorted Set

| Redis 명령어 | Python 메서드 |
|---|---|
| `ZADD key score member` | `r.zadd(key, {member: score})` |
| `ZRANGE ... REV WITHSCORES` | `r.zrevrange(key, 0, -1, withscores=True)` |
| `ZINCRBY key n member` | `r.zincrby(key, n, member)` |
| `ZSCORE key member` | `r.zscore(key, member)` |
| `ZREVRANK key member` | `r.zrevrank(key, member)` |

In [10]:
r.delete('ranking:weekly')

# r.zadd(key, {member: score}): member와 score(점수)를 함께 저장합니다
# Sorted Set은 score 기준으로 자동 정렬되므로 랭킹 구현에 최적입니다
r.zadd('ranking:weekly', {
    'user:1': 980, 'user:2': 1250, 'user:3': 730,
    'user:4': 1500, 'user:5': 840,
})

5

In [11]:
# r.zrevrange(key, start, end, withscores=True): 점수 높은 순(내림차순)으로 조회
#   → withscores=True 를 주면 (member, score) 튜플 리스트를 반환합니다
scores = r.zrevrange('ranking:weekly', 0, -1, withscores=True)

df = pd.DataFrame(scores, columns=['사용자', '점수'])
df.insert(0, '순위', range(1, len(df) + 1))  # 1위부터 순위 번호 추가
df['점수'] = df['점수'].astype(int)           # score가 float로 오므로 int로 변환

df.head()

,순위,사용자,점수
0,1,user:4,1500
1,2,user:2,1250
2,3,user:1,980
3,4,user:5,840
4,5,user:3,730


In [12]:
# r.zincrby(key, amount, member): 특정 멤버의 score를 amount만큼 원자적으로 증가
r.zincrby('ranking:weekly', 400, 'user:5')   # user:5 점수 840 → 1240으로 증가

# r.zscore(key, member):          멤버의 현재 score 반환 (없으면 None)
score = int(r.zscore('ranking:weekly', 'user:5'))
print(f'user:5 점수: {score}')          # 1240

# r.zrevrank(key, member):        내림차순 기준 순위 반환 (0=1위)
rank = r.zrevrank('ranking:weekly', 'user:5')  # 0부터 시작 (0이면 1위)

print(f'user:5 순위 (0=1위): {rank}')   # 2

user:5 점수: 1240
user:5 순위 (0=1위): 2


## 5. Hash

| Redis 명령어 | Python 메서드 |
|---|---|
| `HSET key f1 v1 f2 v2` | `r.hset(key, mapping={f1: v1, f2: v2})` |
| `HGET key field` | `r.hget(key, field)` |
| `HGETALL key` | `r.hgetall(key)` |
| `HINCRBY key field n` | `r.hincrby(key, field, n)` |
| `HDEL key field` | `r.hdel(key, field)` |

In [13]:
r.delete('product:1001')

# r.hset(key, mapping={...}): 여러 필드를 한 번에 저장 (Python dict와 유사)
# Hash는 관련된 정보(상품 정보, 사용자 프로필 등)를 하나의 키로 묶을 때 유용합니다
r.hset('product:1001', mapping={
    'name': '무선 키보드', 'price': '49000',
    'stock': '20', 'category': 'keyboard',
})

4

In [14]:
# r.hgetall(key): Hash의 모든 필드와 값을 dict로 반환합니다
data = r.hgetall('product:1001')

# .items()로 (필드명, 값) 쌍의 리스트를 만들어 DataFrame으로 시각화합니다
pd.DataFrame(data.items(), columns=['필드', '값'])

,필드,값
0,name,무선 키보드
1,price,49000
2,stock,20
3,category,keyboard


In [15]:
# r.hincrby(key, field, amount): Hash 내 특정 필드의 숫자 값을 원자적으로 증가/감소
#   → 음수를 주면 감소합니다 (-1이면 재고 1개 차감)
r.hincrby('product:1001', 'stock', -1)    # 재고 1 감소 (20 → 19)

# r.hdel(key, field): 특정 필드만 삭제 (키 전체가 아닌 필드 단위 삭제)
r.hdel('product:1001', 'category')        # category 필드 삭제

1

In [16]:
data = r.hgetall('product:1001')
pd.DataFrame(data.items(), columns=['필드', '값'])

,필드,값
0,name,무선 키보드
1,price,49000
2,stock,19


## 실습

1. **String**: `user:1`의 로그인 횟수를 저장하고 3번 증가시키세요.
2. **List**: 장바구니(`cart:user1`)를 만들고 상품 3개를 추가한 뒤, 1개를 꺼내세요.
3. **Set**: 두 사용자의 팔로우 목록을 만들고 공통 팔로우(`SINTER`)를 구하세요.
4. **Sorted Set**: 상품 판매량 랭킹을 만들고 상위 3개를 DataFrame으로 출력하세요.
5. **Hash**: 사용자 프로필을 저장하고 포인트(`points`)를 `100` 증가시키세요.

### 1. String: 로그인 횟수 카운터

In [17]:
# r.set으로 초기값 0을 저장한 뒤, r.incr을 3번 호출해 3으로 만듭니다
r.set('login:count:user:1', 0)
r.incr('login:count:user:1')   # 0 → 1
r.incr('login:count:user:1')   # 1 → 2
r.incr('login:count:user:1')   # 2 → 3

count = r.get('login:count:user:1')
print(f'user:1 로그인 횟수: {count}')   # 3

user:1 로그인 횟수: 3


### 2. List: 장바구니에 상품 3개 추가, 1개 꺼내기

In [18]:
# r.delete로 기존 장바구니를 초기화하고, rpush로 오른쪽에 상품을 추가합니다
r.delete('cart:user1')

r.rpush('cart:user1', 'item:A', 'item:B', 'item:C')
print(f'장바구니: {r.lrange("cart:user1", 0, -1)}')   # ['item:A', 'item:B', 'item:C']

장바구니: ['item:A', 'item:B', 'item:C']


In [19]:
# lpop으로 왼쪽(가장 먼저 추가된 상품)을 꺼냅니다
item = r.lpop('cart:user1')
print(f'꺼낸 상품: {item}')                              # item:A
print(f'남은 장바구니: {r.lrange("cart:user1", 0, -1)}')

꺼낸 상품: item:A
남은 장바구니: ['item:B', 'item:C']


### 3. Set: 두 사용자의 팔로우 목록 및 공통 팔로우

In [20]:
r.delete('follow:user1', 'follow:user2')

# sadd로 각 사용자의 팔로우 목록을 저장하고, smembers로 전체 조회합니다
r.sadd('follow:user1', 'user:A', 'user:B', 'user:C')
r.sadd('follow:user2', 'user:B', 'user:C', 'user:D')

print(f'user1 팔로우: {r.smembers("follow:user1")}')
print(f'user2 팔로우: {r.smembers("follow:user2")}')

user1 팔로우: {'user:A', 'user:C', 'user:B'}
user2 팔로우: {'user:D', 'user:C', 'user:B'}


In [21]:
# sinter로 두 집합의 교집합(공통 팔로우)을 구합니다
common = r.sinter('follow:user1', 'follow:user2')
print(f'공통 팔로우: {common}')   # {'user:B', 'user:C'}

공통 팔로우: {'user:C', 'user:B'}


### 4. Sorted Set: 상품 판매량 랭킹 상위 3개 DataFrame 출력

In [22]:
r.delete('sales:ranking')

# zadd에 딕셔너리 형태 {member: score}로 여러 항목을 한 번에 추가할 수 있습니다
r.zadd('sales:ranking', {
    'product:A': 450, 'product:B': 320, 'product:C': 780,
    'product:D': 210, 'product:E': 560,
})

5

In [23]:
# zrevrange(key, 0, 2): 점수 내림차순으로 인덱스 0~2(상위 3개)만 조회합니다
top3 = r.zrevrange('sales:ranking', 0, 2, withscores=True)

df = pd.DataFrame(top3, columns=['상품', '판매량'])
df.insert(0, '순위', range(1, len(df) + 1))
df['판매량'] = df['판매량'].astype(int)   # float → int 변환

df.head()

,순위,상품,판매량
0,1,product:C,780
1,2,product:E,560
2,3,product:A,450


### 5. Hash: 사용자 프로필 저장 + 포인트 증가

In [24]:
r.delete('profile:user1')

# hset으로 프로필 전체를 한 번에 저장하고, hincrby로 points 필드만 증가시킵니다
r.hset('profile:user1', mapping={
    'name': '김철수', 'email': 'chulsoo@example.com',
    'level': '3', 'points': '500',
})

# hincrby는 반환값으로 변경 후의 값을 반환합니다 (500 + 100 = 600)
r.hincrby('profile:user1', 'points', 100)

600

In [25]:
data = r.hgetall('profile:user1')
pd.DataFrame(data.items(), columns=['필드', '값'])

,필드,값
0,name,김철수
1,email,chulsoo@example.com
2,level,3
3,points,600
